# Exceptions and File I/O

**Source:** https://labs.acme.byu.edu/PythonEssentials/Exceptions_FileIO/Exceptions_FileIO.html

This lab covers Python's exception handling system, raising and catching exceptions, reading and writing files with context managers, and string formatting.

## 1. Exceptions

Common types: `ValueError`, `TypeError`, `IndexError`, `FileNotFoundError`, `KeyboardInterrupt`.

In [1]:
def positive_sqrt(x):
    if x < 0:
        raise ValueError(f"Input must be non-negative, got {x}")
    return x ** 0.5

print(positive_sqrt(4))
try:
    
    print(positive_sqrt(-1))
except ValueError as e:
    print(f"Caught: {e}")

2.0
Caught: Input must be non-negative, got -1


## 2. Try-Except-Else-Finally

In [2]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    else:

        return result
    finally:
        print("Attempted division")

print(safe_divide(10, 2))
print(safe_divide(10, 0))

Attempted division
5.0
Cannot divide by zero
Attempted division
None


## 3. File I/O with Context Managers

In [3]:
import os, tempfile
tmpdir = tempfile.gettempdir()
sample_path = os.path.join(tmpdir, "acme_sample.txt")

with open(sample_path, 'w') as f:
    f.write("Hello World foo\nbar baz qux\nPython is great\n123 456 789\n")


with open(sample_path, 'r') as f:
    for line in f:
        print(repr(line))

'Hello World foo\n'
'bar baz qux\n'
'Python is great\n'
'123 456 789\n'


---

## Problem 1: Arithmagic Validation

Validate each step of the arithmagic sequence, raising `ValueError` with descriptive messages.

In [4]:
def validate_arithmagic(n1, n2, n3, n4):
    """Validate the four steps of the arithmagic sequence.
    
    Raises ValueError with descriptive messages for any violation.
    Returns the final sum (always 1089 for valid input).
    """
    s1 = str(n1)
    if len(s1) != 3:
        raise ValueError(f"{n1} is not a 3-digit number")
    if abs(int(s1[0]) - int(s1[2])) < 2:
        raise ValueError(f"First and last digits of {n1} must differ by at least 2")
    if n2 != int(s1[::-1]):
        raise ValueError(f"{n2} is not the reverse of {n1} (expected {int(s1[::
            -1])})")
    expected_diff = abs(n1 - n2)
    if n3 != expected_diff:
        raise ValueError(f"{n3} is not the absolute difference of {n1} and {n2} (expected {expected_diff})")
    s3 = str(n3).zfill(3)
    if n4 != int(s3[::-1]):
        raise ValueError(f"{n4} is not the reverse of {n3} (expected {int(s3[::-1])})")
    return n3 + n4

# Test valid cases
print(validate_arithmagic(621, 126, 495, 594))   # 1089
print(validate_arithmagic(835, 538, 297, 792))   # 1089

# Test invalid cases
for bad in [(111, 111, 0, 0), (621, 200, 495, 594)]:
    try:
        validate_arithmagic(*bad)
    except ValueError as e:
        print(f"ValueError: {e}")

1089
1089
ValueError: First and last digits of 111 must differ by at least 2
ValueError: 200 is not the reverse of 621 (expected 126)


## Problem 2: Random Walk Interrupt Handler

In [5]:
import numpy as np

def random_walk(max_iters=1e12):
    """Perform a random walk; catch KeyboardInterrupt and report gracefully."""
    walk = 0
    directions = [1, -1]
    try:
        for i in range(int(max_iters)):
            walk += np.random.choice(directions)
            
    except KeyboardInterrupt:
        print(f"Process interrupted at iteration {i}")
    else:
        print("Process completed")
    return walk

result = random_walk(max_iters=10000)
print(f"Final position: {result}")

Process completed
Final position: -180


## Problems 3 & 4: ContentFilter Class

In [6]:
class ContentFilter:
    """Read a text file and provide case, reverse, and transpose transformations.

    Attributes:
        filename (str): path to the source file.
        contents (str): full text of the file.
    """

    def __init__(self, filename):
        """Open file, prompting until a valid file is provided."""
        while True:
            try:
                with open(filename, 'r') as f:
                    self.contents = f.read()
                self.filename = filename
                break
            except (FileNotFoundError, TypeError, UnicodeDecodeError, OSError) as e:
                print(f"{type(e).__name__}: {e}")
                filename = input("Please enter a valid file name: ")

    def uniform(self, outfile, mode='w', case='upper'):
        
        """Write file contents with uniform case.

        Parameters:
            outfile (str): destination file path.
            mode (str): write mode ('w' or 'a'). Raises ValueError if invalid.
            case (str): 'upper' or 'lower'.
        """
        if mode not in ('w', 'a', 'x'):
            raise ValueError(f"Invalid mode: '{mode}'")
        text = self.contents.upper() if case == 'upper' else self.contents.lower()
        with open(outfile, mode) as f:
            f.write(text)

    def reverse(self, outfile, mode='w', unit='line'):
        """Write file with lines or words reversed.

        Parameters:
            outfile (str): destination file path.
            mode (str): write mode. Raises ValueError if invalid.
            unit (str): 'line' reverses line order; 'word' reverses words per line.
        """
        if mode not in ('w', 'a', 'x'):
            raise ValueError(f"Invalid mode: '{mode}'")
        lines = self.contents.splitlines()
        if unit == 'line':
            result = '\n'.join(reversed(lines))
        elif unit == 'word':
            result = '\n'.join(' '.join(reversed(line.split())) for line in lines)
        else:
            raise ValueError(f"unit must be 'line' or 'word', got '{unit}'")
        with open(outfile, mode) as f:
            f.write(result)

    def transpose(self, outfile, mode='w'):
        """Treat words as a matrix and transpose rows/columns.

        Parameters:
            outfile (str): destination file path.
            mode (str): write mode. Raises ValueError if invalid.
        """
        if mode not in ('w', 'a', 'x'):
            raise ValueError(f"Invalid mode: '{mode}'")
        lines = [line.split() for line in self.contents.splitlines() if line.strip()]
        result = '\n'.join(' '.join(row) for row in zip(*lines))
        with open(outfile, mode) as f:
            f.write(result)

    def __str__(self):
        """Return statistics about the file contents."""
        return (f"Source file:\t\t{self.filename}\n"
                f"Total characters:\t{len(self.contents)}\n"
                f"Alphabetic characters:\t{sum(c.isalpha() for c in self.contents)}\n"
                f"Numerical characters:\t{sum(c.isdigit() for c in self.contents)}\n"
                f"Whitespace characters:\t{sum(c.isspace() for c in self.contents)}\n"
                f"Number of lines:\t{self.contents.count(chr(10))}")


cf = ContentFilter(sample_path)
print(cf)
print()

out_upper   = os.path.join(tmpdir, "upper.txt")
out_rev     = os.path.join(tmpdir, "reversed.txt")
out_transp  = os.path.join(tmpdir, "transposed.txt")

cf.uniform(out_upper, case='upper')
cf.reverse(out_rev, unit='word')
cf.transpose(out_transp)

print("--- Reversed by word ---")
with open(out_rev) as f:
    print(f.read())

print("--- Transposed ---")
with open(out_transp) as f:
    print(f.read())

Source file:		/tmp/acme_sample.txt
Total characters:	56
Alphabetic characters:	35
Numerical characters:	9
Whitespace characters:	12
Number of lines:	4

--- Reversed by word ---
foo World Hello
qux baz bar
great is Python
789 456 123
--- Transposed ---
Hello bar Python 123
World baz is 456
foo qux great 789
